# 💊 Side Effect Extraction from Text using SIDER and openFDA
This notebook combines side effect dictionaries from SIDER and openFDA drug labels,
and uses `spaCy` to extract mentions of adverse events from clinical or biomedical text.

In [ ]:
# 📦 Install required libraries (uncomment if not installed)
# !pip install pandas spacy requests
# !python -m spacy download en_core_web_sm

In [1]:
import pandas as pd
import spacy as spc
import scispacy as scspcy
# Multiple ways to get and work with home directory
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
import requests
import re
import math
from collections import Counter
from IPython.display import Audio, display
import random
import time

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")
#files = os.listdir(os.getcwd())
#print(files)

# Step 3: Download and load the model
try:
    nlp = spc.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi

Contents of home directory:
Current working directory: /Users/naveendeevi/Datasets
Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [2]:
import pandas as pd
import requests
import spacy
from spacy.matcher import PhraseMatcher
from pathlib import Path

## 🔹 Step 1: Load Side Effect Terms from SIDER
Make sure to download `meddra_all_se.tsv` from http://sideeffects.embl.de/download/

In [34]:
sider_path = "meddra_all_se.tsv"  # Update path if needed
df = pd.read_csv(sider_path, sep='\t', header=None)
df.columns = ['stitch_flat', 'stitch_stereo', 'umls_cui', 'side_effect_code', 'method','side_effect_name']
sider_terms = set(df['side_effect_name'].str.lower().unique())

In [138]:
sider_terms.update(['suicidal','suicide','gastrointestinal','confusion','libido','weakness','panic attack','aggressiveness',
                   'dizziness','seizures','blurred vision','apnea','restlessness','tremors','hallucinations','delusion',
                   'sleep disturbance','birth defects','miscarriage','falls','factures'])

## 🔹 Step 2: Get Additional Side Effects from openFDA Label
We’ll pull adverse reactions from drug label fields like `adverse_reactions`, `warnings`, etc.

In [3]:
import requests

def get_openfda_ae_terms(drug_name):
    fields_to_try = ["openfda.generic_name", "openfda.brand_name"]
    ae_fields = ['adverse_reactions', 'warnings', 'boxed_warning', 'precautions']

    for field in fields_to_try:
        url = f"https://api.fda.gov/drug/label.json?search={field}:\"{drug_name}\"&limit=1"
        response = requests.get(url)

        if response.status_code == 200:
            try:
                result = response.json()['results'][0]
                
                extracted_text = []
                for ae_field in ae_fields:
                    val = result.get(ae_field, '')
                    if isinstance(val, list):
                        extracted_text.extend(val)
                    elif isinstance(val, str):
                        extracted_text.append(val)

                combined = ' '.join(extracted_text)
                return [sent.strip().lower() for sent in combined.split('.') if len(sent.strip()) > 3]
            except Exception as e:
                print(f"Error parsing results: {e}")
                return []
        elif response.status_code != 404:
            print(f"Request failed with status code {response.status_code}")
            return []
    
    print("No matching record found.")
    return []

# Example usage
phrases = get_openfda_ae_terms("mirtazapine")
#print(phrases)  # Print just first few for sanity check


In [139]:
#print(phrases)

## 🔹 Step 3: Match Side Effects in Input Text using spaCy

In [126]:
# Install the small English model for spaCy
#!python -m spacy download en_core_web_sm

In [127]:
#phrases

In [131]:
nlp = spacy.load("en_core_web_sm")
#nlp = spacy.load("en_ner_bc5cdr_md")
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

all_terms = list(sider_terms.union(phrases))
#print(all_terms)

patterns = [nlp.make_doc(term) for term in all_terms if isinstance(term, str) and 1 <= len(term.split()) <= 5]
matcher.add("SIDE_EFFECT", patterns)

def extract_side_effects(text):
    doc = nlp(text)
    matches = matcher(doc)
    return list(set(doc[start:end].text.lower() for _, start, end in matches))

<class 'spacy.matcher.phrasematcher.PhraseMatcher'>


In [129]:
print(sorted(extract_side_effects(sample_text)))

['4)]', 'abdominal pain', 'abnormal dreams', 'abnormal ejaculation', 'acne', 'adverse reaction', 'agitation', 'agranulocytosis', 'akathisia', 'alopecia', 'amnesia', 'angina pectoris', 'anorexia', 'anxiety', 'apathy', 'aphasia', 'aphthous stomatitis', 'arrhythmia', 'arthralgia', 'arthritis', 'asphyxia', 'asthenia', 'asthma', 'ataxia', 'atrial arrhythmia', 'back pain', 'blepharitis', 'bone pain', 'bradycardia', 'breast disorder', 'breast engorgement', 'breast enlargement', 'breast pain', 'bronchitis', 'bursitis', 'cardiomegaly', 'cellulitis', 'chest pain', 'chills', 'cholecystitis', 'colitis', 'conjunctivitis', 'constipation', 'convulsion', 'coordination abnormal', 'cough', 'cough increased', 'cystitis', 'deafness', 'dehydration', 'delirium', 'dementia', 'dependence', 'depression', 'dermatitis', 'diabetes mellitus', 'diplopia', 'dizziness', 'drug dependence', 'dry mouth', 'dry skin', 'dysarthria', 'dyskinesia', 'dystonia', 'dysuria', 'ear pain', 'embolus', 'eosinophilia', 'epistaxis', 'e

## 🔹 Step 4: Test the Extractor

In [59]:
sample_text = """The following adverse reactions are described in more detail in other sections of the prescribing information:

• Hypersensitivity [see Contraindications (4)]

• Suicidal Thoughts and Behaviors [see Warnings and Precautions (5.1)]

• Agranulocytosis [see Warnings and Precautions (5.2)]

• Serotonin Syndrome [see Contraindications (4), Warnings and Precautions (5.3), Drug Interactions (7)]

• Angle-Closure Glaucoma [see Warnings and Precautions (5.4)]

• QT Prolongation and Torsades de Pointes [see Warnings and Precautions (5.5)]

• Drug Reaction with Eosinophilia and Systemic Symptoms (DRESS) [see Warnings and Precautions (5.6)]

• Increased Appetite and Weight Gain [see Warnings and Precautions (5.7)]

• Somnolence [see Warnings and Precautions (5.8)]

• Activation of Mania or Hypomania [see Warnings and Precautions (5.9)]

• Seizures [see Warnings and Precautions (5.10)]

• Elevated Cholesterol and Triglycerides [see Warnings and Precautions (5.11)]

• Hyponatremia [see Warnings and Precautions (5.12)]

• Transaminase Elevations [see Warnings and Precautions (5.13)]

• Discontinuation Syndrome [see Warnings and Precautions (5.14)]

• Use in Patients with Concomitant Illness [see Warnings and Precautions (5.15)]

6.1 Clinical Trial Experience
Because clinical trials are conducted under widely varying conditions, adverse reaction rates observed in the clinical trials of a drug cannot be directly compared to rates in the clinical trials of another drug and may not reflect the rates observed in practice.

The data described below are from clinical trials in which mirtazapine tablets were administered to 2,796 patients in phase 2 and 3 clinical studies. The trials consisted of double-blind controlled and open-label studies, inpatient and outpatient studies, fixed dose, and titration studies.

Adverse Reactions Leading to Discontinuation of Treatment

Approximately 16% of the 453 patients who received mirtazapine tablets in U.S. 6-week placebo-controlled clinical trials discontinued treatment due to an adverse reaction, compared to 7% of the 361 placebo-treated patients in those studies. The most common reactions leading to discontinuation (≥ 1% and at a rate at least twice that of placebo) are included in Table 2:

Table 2: Adverse Reactions (≥ 1% and at least twice placebo) Leading to Discontinuation of Mirtazapine Tablets in 6-Week Clinical Trials in Patients with MDD

Mirtazapine Tablets
(n=453)	Placebo
(n=361)
Somnolence	10.4%	2.2%
Nausea	1.5%	0%
Common Adverse Reactions

The most common adverse reactions (≥ 5% and twice placebo) associated with the use of mirtazapine tablets are listed in Table 3.

Table 3: Adverse Reactions (≥ 5% and twice placebo) in 6-Week U.S. Clinical Trials of Mirtazapine Tablets in Patients with MDD

Mirtazapine Tablets   (n=453) 	Placebo   (n=361) 
Somnolence	54%	18%
Increased Appetite	17%	2%
Weight Gain	12%	2%
Dizziness	7%	3%
Table 4 enumerates adverse reactions that occurred in ≥ 1% of mirtazapine tablets-treated patients and were more frequent than the placebo-treated patients, who participated in 6-week, U.S. placebo-controlled trials in which patients were dosed in a range of 5 to 60 mg/day. This table shows the percentage of patients in each group who had at least 1 episode of an adverse reaction at some time during their treatment.

Table 4: Adverse Reactions (≥ 1% and greater than placebo) in 6-Week U.S. Clinical Studies of Mirtazapine Tablets in Patients with MDD

 	Mirtazapine Tablets   (n=453) 	Placebo   (n=361) 
Body as a Whole 
Asthenia	8%	5%
Flu Syndrome	5%	3%
Back Pain	2%	1%
Digestive System 
Dry Mouth	25%	15%
Increased Appetite	17%	2%
Constipation	13%	7%
Metabolic and Nutritional Disorders 
Weight Gain	12%	2%
Peripheral Edema	2%	1%
Edema	1%	0%
Musculoskeletal System 
Myalgia	2%	1%
Nervous System 
Somnolence	54%	18%
Dizziness	7%	3%
Abnormal Dreams	4%	1%
Thinking Abnormal	3%	1%
Tremor	2%	1%
Confusion	2%	0%
Respiratory System 
Dyspnea	1%	0%
Urogenital System 
Urinary Frequency	2%	1%
ECG Changes

The electrocardiograms for 338 patients who received mirtazapine tablets and 261 patients who received placebo in 6-week, placebo-controlled trials were analyzed. Mirtazapine tablets was associated with a mean increase in heart rate of 3.4 bpm, compared to 0.8 bpm for placebo. The clinical significance of these changes is unknown.

Other Adverse Reactions Observed During the Premarketing Evaluation of Mirtazapine Tablets

The following list does not include reactions: 1) already listed in previous tables or elsewhere in labeling, 2) for which a drug cause was remote, 3) which were so general or excessively specific so as to be uninformative, 4) which were not considered to have significant clinical implications, or 5) which occurred at a rate equal to or less than placebo.

Adverse reactions are categorized by body system according to the following definitions: frequent adverse reactions are those occurring in at least 1/100 patients; infrequent adverse reactions are those occurring in 1/100 to 1/1,000 patients; rare adverse reactions are those occurring in fewer than 1/1,000 patients.

Body as a Whole: frequent: malaise, abdominal pain, abdominal syndrome acute; infrequent: chills, fever, face edema, ulcer, photosensitivity reaction, neck rigidity, neck pain, abdomen enlarged; rare: cellulitis, chest pain substernal.

Cardiovascular System: frequent: hypertension, vasodilatation; infrequent: angina pectoris, myocardial infarction, bradycardia, ventricular extrasystoles, syncope, migraine, hypotension; rare: atrial arrhythmia, bigeminy, vascular headache, pulmonary embolus, cerebral ischemia, cardiomegaly, phlebitis, left heart failure.

Digestive System: frequent: vomiting, anorexia; infrequent: eructation, glossitis, cholecystitis, nausea and vomiting, gum hemorrhage, stomatitis, colitis, liver function tests abnormal; rare: tongue discoloration, ulcerative stomatitis, salivary gland enlargement, increased salivation, intestinal obstruction, pancreatitis, aphthous stomatitis, cirrhosis of liver, gastritis, gastroenteritis, oral moniliasis, tongue edema.

Endocrine System: rare: goiter, hypothyroidism.

Hemic and Lymphatic System: rare: lymphadenopathy, leukopenia, petechia, anemia, thrombocytopenia, lymphocytosis, pancytopenia.

Metabolic and Nutritional Disorders: frequent: thirst; infrequent: dehydration, weight loss; rare: gout, SGOT increased, healing abnormal, acid phosphatase increased, SGPT increased, diabetes mellitus, hyponatremia.

Musculoskeletal System: frequent: myasthenia, arthralgia; infrequent: arthritis, tenosynovitis; rare: pathologic fracture, osteoporosis fracture, bone pain, myositis, tendon rupture, arthrosis, bursitis.

Nervous System: frequent: hypesthesia, apathy, depression, hypokinesia, vertigo, twitching, agitation, anxiety, amnesia, hyperkinesia, paresthesia; infrequent: ataxia, delirium, delusions, depersonalization, dyskinesia, extrapyramidal syndrome, libido increased, coordination abnormal, dysarthria, hallucinations, manic reaction, neurosis, dystonia, hostility, reflexes increased, emotional lability, euphoria, paranoid reaction; rare: aphasia, nystagmus, akathisia (psychomotor restlessness), stupor, dementia, diplopia, drug dependence, paralysis, grand mal convulsion, hypotonia, myoclonus, psychotic depression, withdrawal syndrome, serotonin syndrome.

Respiratory System: frequent: cough increased, sinusitis; infrequent: epistaxis, bronchitis, asthma, pneumonia; rare: asphyxia, laryngitis, pneumothorax, hiccup.

Skin and Appendages: frequent: pruritus, rash; infrequent: acne, exfoliative dermatitis, dry skin, herpes simplex, alopecia; rare: urticaria, herpes zoster, skin hypertrophy, seborrhea, skin ulcer.

Special Senses: infrequent: eye pain, abnormality of accommodation, conjunctivitis, deafness, keratoconjunctivitis, lacrimation disorder, angle-closure glaucoma, hyperacusis, ear pain; rare: blepharitis, partial transitory deafness, otitis media, taste loss, parosmia.

Urogenital System: frequent: urinary tract infection; infrequent: kidney calculus, cystitis, dysuria, urinary incontinence, urinary retention, vaginitis, hematuria, breast pain, amenorrhea, dysmenorrhea, leukorrhea, impotence; rare: polyuria, urethritis, metrorrhagia, menorrhagia, abnormal ejaculation, breast engorgement, breast enlargement, urinary urgency.

6.2 Postmarketing Experience
The following adverse reactions have been identified during post-approval use of mirtazapine tablets. Because these reactions are reported voluntarily from a population of uncertain size, it is not always possible to reliably estimate their frequency or establish a causal relationship to drug exposure.

Cardiac disorders: ventricular arrhythmia (Torsades de Pointes)

Endocrine disorders: hyperprolactinemia (and related symptoms, e.g., galactorrhea and gynecomastia)

Musculoskeletal and connective tissue disorders: increased creatine kinase blood levels and rhabdomyolysis

Psychiatric disorders: somnambulism (ambulation and other complex behaviors out of bed)

Reproductive system and breast disorder: priapism

Skin and subcutaneous tissue disorders: severe skin reactions, including DRESS, Stevens-Johnson syndrome, bullous dermatitis, erythema multiforme and toxic epidermal necrolysis"""

In [61]:

text = "The patient experienced suicidal ideation and mild tremor after taking fluoxetine."
print(sorted(extract_side_effects(sample_text)))

['abdominal pain', 'abnormal dreams', 'abnormal ejaculation', 'acne', 'adverse reaction', 'agitation', 'agranulocytosis', 'akathisia', 'alopecia', 'amnesia', 'angina pectoris', 'anorexia', 'anxiety', 'apathy', 'aphasia', 'aphthous stomatitis', 'arrhythmia', 'arthralgia', 'arthritis', 'asphyxia', 'asthenia', 'asthma', 'ataxia', 'atrial arrhythmia', 'back pain', 'blepharitis', 'bone pain', 'bradycardia', 'breast disorder', 'breast engorgement', 'breast enlargement', 'breast pain', 'bronchitis', 'bursitis', 'cardiomegaly', 'cellulitis', 'chest pain', 'chills', 'cholecystitis', 'colitis', 'conjunctivitis', 'constipation', 'convulsion', 'coordination abnormal', 'cough', 'cough increased', 'cystitis', 'deafness', 'dehydration', 'delirium', 'dementia', 'dependence', 'depression', 'dermatitis', 'diabetes mellitus', 'diplopia', 'dizziness', 'drug dependence', 'dry mouth', 'dry skin', 'dysarthria', 'dyskinesia', 'dystonia', 'dysuria', 'ear pain', 'embolus', 'eosinophilia', 'epistaxis', 'eructati

In [143]:
len(sorted(all_terms))
if 'sedation' in all_terms: 
    print('true')

true


In [4]:
def get_openfda_ae_terms(drug_name):
    fields_to_try = ["openfda.generic_name", "openfda.brand_name"]
    ae_fields = ['adverse_reactions', 'warnings', 'boxed_warning', 'precautions', 'warnings_and_cautions']

    for field in fields_to_try:
        url = f"https://api.fda.gov/drug/label.json?search={field}:\"{drug_name}\"&limit=1"
        response = requests.get(url)
        print(url)
        if response.status_code == 200:
            try:
                result = response.json()['results'][0]
                
                extracted_text = []
                for ae_field in ae_fields:
                    val = result.get(ae_field, '')
                    if isinstance(val, list):
                        extracted_text.extend(val)
                    elif isinstance(val, str):
                        extracted_text.append(val)

                combined = ' '.join(extracted_text)
                return [sent.strip().lower() for sent in combined.split('.') if len(sent.strip()) > 3]
            except Exception as e:
                #print(f"Error parsing results: {e}")
                return None
        elif response.status_code != 404:
            #print(f"Request failed with status code {response.status_code}")
            return None
    
    #print("No matching record found.")
    return None


In [5]:
get_openfda_ae_terms('alprazolam')

https://api.fda.gov/drug/label.json?search=openfda.generic_name:"alprazolam"&limit=1


['6 adverse reactions the following clinically significant adverse reactions are described elsewhere in the labeling: risks from concomitant use with opioids [see warnings and precautions (5',
 '1) ] abuse, misuse, and addiction [see warnings and precautions (5',
 '2) ] dependence and withdrawal reactions [see warnings and precautions (5',
 '3) ] effects on driving and operating machinery [see warnings and precautions (5',
 '4) ] patients with depression [see warnings and precautions (5',
 '6) ] neonatal sedation and withdrawal syndrome [see warnings and precautions (5',
 '8) ] risks in patients with impaired respiratory function [see warnings and precautions (5',
 '9) ] the most common adverse reactions reported in clinical trials for generalized anxiety disorder and panic disorder (incidence > 5% and at least twice that of placebo) include: impaired coordination, hypotension, dysarthria, and increased libido',
 '1 ) to report suspected adverse reactions, contact aurobindo pharma usa,

In [136]:
def get_openfda_ae_terms_by_rxcui(rxcui):
    ae_fields = ['adverse_reactions', 'warnings', 'boxed_warning', 'precautions']

    url = f"https://api.fda.gov/drug/label.json?search=openfda.rxcui:{rxcui}&limit=1"
    response = requests.get(url)
    print(url)
    if response.status_code == 200:
        try:
            result = response.json()['results'][0]

            extracted_text = []
            for ae_field in ae_fields:
                val = result.get(ae_field, '')
                if isinstance(val, list):
                    extracted_text.extend(val)
                elif isinstance(val, str):
                    extracted_text.append(val)

            combined = ' '.join(extracted_text)
            return [sent.strip().lower() for sent in combined.split('.') if len(sent.strip()) > 3]
        except Exception as e:
            #print(f"Error parsing results: {e}")
            return None
    elif response.status_code != 404:
        #print(f"Request failed with status code {response.status_code}")
        return None

    #print("No matching record found.")
    return None


In [137]:
get_openfda_ae_terms_by_rxcui('1000048')

https://api.fda.gov/drug/label.json?search=openfda.rxcui:1000048&limit=1


['adverse reactions note: some of the adverse reactions noted below have not been specifically reported with doxepin hydrochloride use',
 'however, due to the close pharmacological similarities among the tricyclics, the reactions should be considered when prescribing doxepin hydrochloride',
 'anticholinergic effects: dry mouth, blurred vision, constipation and urinary retention have been reported',
 'if they do not subside with continued therapy or become severe, it may be necessary to reduce the dosage',
 'central nervous system effects: drowsiness is the most commonly noticed side effect',
 'this tends to disappear as therapy is continued',
 'other infrequently reported cns side effects are confusion, disorientation, hallucinations, numbness, paresthesias, ataxia, extrapyramidal symptoms, seizures, tardive dyskinesia and tremor',
 'cardiovascular: cardiovascular effects including hypotension, hypertension and tachycardia have been reported occasionally',
 'allergic: skin rash, edema,

In [144]:
def all_se_drugs(text1,text2,text3):
    all_se = [text1].append([text2])
    all_se = all_se.append([text3])
    all_se = [set(all_se)]
    return all_se

In [148]:
text1 = adverse reaction, agitation, amnesia, anxiety, ataxia, chronic hepatitis, cleft palate, coma, constipation, crying, death, dependence, depression, diplopia, dizziness, drowsiness, drug abuse, dry mouth, dysarthria, encephalopathy, fatigue, fibrosis, headache, hepatic encephalopathy, hepatic fibrosis, hepatic insufficiency, hepatitis, hyperreflexia, hypotension, hypothermia, hypotonia, incontinence, injury, insomnia, irritability, jaundice, lethargy, muscle spasticity, nausea, neutropenia, overdose, oversedation, poisoning, pregnancy, rage, respiratory depression, respiratory insufficiency, salivation, sedation, slurred speech, suicidal, tremor, urinary retention, vertigo, withdrawal syndrome

SyntaxError: invalid syntax (3578457181.py, line 1)

['adverse reaction, agitation, amnesia, anxiety, ataxia, chronic hepatitis, cleft palate, coma, constipation, crying, death, dependence, depression, diplopia, dizziness, drowsiness, drug abuse, dry mouth, dysarthria, encephalopathy, fatigue, fibrosis, headache, hepatic encephalopathy, hepatic fibrosis, hepatic insufficiency, hepatitis, hyperreflexia, hypotension, hypothermia, hypotonia, incontinence, injury, insomnia, irritability, jaundice, lethargy, muscle spasticity, nausea, neutropenia, overdose, oversedation, poisoning, pregnancy, rage, respiratory depression, respiratory insufficiency, salivation, sedation, slurred speech, suicidal, tremor, urinary retention, vertigo, withdrawal syndrome']

In [149]:
df = pd.read_csv('test_2.csv')

In [152]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'rxcui', 'medication_name', 'dose',
       'ingredient_name', 'indication_terms_fda', 'coalesced_class_name',
       'SE_rxcui', 'SE_med_nm', 'SE_ing_nm'],
      dtype='object')

In [156]:
def make_merge_unique_words_function(cols_to_merge):
    def merge_unique_words(row):
        words = set()
        for col in cols_to_merge:
            val = row[col]
            if isinstance(val, str):
                words.update(val.lower().split())
        return ' '.join(sorted(words))
    return merge_unique_words


In [172]:
cols = ['SE_rxcui', 'SE_med_nm', 'SE_ing_nm']
merge_func = make_merge_unique_words_function(cols)
df['Side_Effects_Combined'] = df.apply(merge_func, axis=1)
#df['combined'] = df[['SE_rxcui','SE_med_nm','SE_ing_nm']].apply(make_merge_unique_words_function,axis=1)

In [173]:
df.columns

Index(['rxcui', 'medication_name', 'dose', 'ingredient_name',
       'indication_terms_fda', 'coalesced_class_name', 'SE_rxcui', 'SE_med_nm',
       'SE_ing_nm', 'combined', 'Side_Effects_Combined'],
      dtype='object')

In [170]:
df.drop(['Unnamed: 0.1','Unnamed: 0'],axis=1,inplace=True)

In [175]:
cols_2 = ['rxcui','medication_name','dose','ingredient_name','indication_terms_fda','coalesced_class_name','Side_Effects_Combined']
df_2 = df[cols_2]

In [176]:
df_2.head(4)

,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,coalesced_class_name,Side_Effects_Combined
0,1000048.0,doxepin 10 mg oral capsule,10 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
1,1000048.0,doxepin 10 mg oral capsule,20 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
2,1000048.0,doxepin 10 mg oral capsule,30 mg,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
3,1000048.0,doxepin 10 mg oral capsule,NaN,doxepin,"[('insomnia', 2)]",Non-selective monoamine reuptake inhibitors (A...,"abdominal abnormal abnormal, accident, adenoca..."
